In [5]:
pip install fuzzywuzzy

In [6]:
#Loading the necessary library
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from fuzzywuzzy import fuzz
from nltk.corpus import stopwords

In [7]:
#mounting the dataset through the google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
#Loading the dataset
movies = pd.read_csv('dataset/movie.csv')

In [9]:
#checking the null values in the dataset
movies.isnull().sum()
#keywords.isnull().sum()

,0
Unnamed: 0,0
adult,0
backdrop_path,200
genres,0
id,0
imdb_id,216
original_language,0
overview,72
popularity,0
poster_path,23


In [10]:
#displaying the top 5 columns of the dataset
movies.head()

,Unnamed: 0,adult,backdrop_path,genres,id,imdb_id,original_language,overview,popularity,poster_path,release_date,runtime,tagline,title,vote_average,vote_count,actors,director,keywords
0,0,False,/jXJxMcVoEuXzym3vFnjqDW4ifo6.jpg,"['Action', 'Adventure', 'Fantasy']",572802,tt9663764,en,"Black Manta, still driven by the need to aveng...",5841.384,/7lTnXOy0iNtBAdRP3TZvaKJ77F6.jpg,2023-12-20,124,The tide is turning.,Aquaman and the Lost Kingdom,7.013,1156,"['Jason Momoa', 'Patrick Wilson', 'Yahya Abdul...",['James Wan'],"['superhero', 'secret society', 'half-brother'..."
1,1,False,/yOm993lsJyPmBodlYjgpPwBjXP9.jpg,"['Comedy', 'Family', 'Fantasy']",787699,tt6166392,en,Willy Wonka – chock-full of ideas and determin...,2333.391,/qhb1qOilapbapxWQn9jtRCMwXJF.jpg,2023-12-06,117,Every good thing in this world started with a ...,Wonka,7.200,1660,"['Timothée Chalamet', 'Calah Lane', 'Keegan-Mi...",['Paul King'],"['chocolate', 'musical', 'prequel', 'duringcre..."
2,2,False,/ehumsuIBbgAe1hg343oszCLrAfI.jpg,"['Animation', 'Family', 'Fantasy', 'Adventure']",1022796,tt11304740,en,"Asha, a sharp-witted idealist, makes a wish so...",1662.711,/AcoVfiv1rrWOmAdpnAMnM56ki19.jpg,2023-11-13,95,Be careful what you wish for.,Wish,6.638,538,"['Ariana DeBose', 'Chris Pine', 'Alan Tudyk', ...",['Chris Buck'],"['friendship', 'musical', 'computer animation'..."
3,3,False,/meyhnvssZOPPjud4F1CjOb4snET.jpg,"['Animation', 'Adventure', 'Comedy', 'Family',...",940551,tt6495056,en,After a migrating duck family alights on their...,1486.310,/ldfCF9RhR40mppkzmftxapaHeTo.jpg,2023-12-06,83,Odd ducks welcome.,Migration,7.830,390,"['Kumail Nanjiani', 'Elizabeth Banks', 'Caspar...",['Benjamin Renner'],"['duck', 'migration', 'flight', 'anthropomorph..."
4,4,False,/tLxjbT5ROZRwYcpNT3nfQbqkApk.jpg,"['Science Fiction', 'Adventure', 'Action']",609681,tt10676048,en,"Carol Danvers, aka Captain Marvel, has reclaim...",1397.918,/9GBhzXMFjgcZ3FdR9w3bUMMTps5.jpg,2023-11-08,105,Higher. Further. Faster. Together.,The Marvels,6.415,1328,"['Brie Larson', 'Teyonah Parris', 'Iman Vellan...",['Nia DaCosta'],"['hero', 'superhero', 'space travel', 'based o..."


In [11]:
#displaying the last 5 columns of the dataset
movies.tail()

,Unnamed: 0,adult,backdrop_path,genres,id,imdb_id,original_language,overview,popularity,poster_path,release_date,runtime,tagline,title,vote_average,vote_count,actors,director,keywords
9995,9995,False,/xSN7sVwIC6ExQZVMIUMjm4Tl2TN.jpg,"['Drama', 'War']",10178,tt0046816,en,When a US Naval captain shows signs of mental ...,12.677,/vuO4Z3wOWVlhq35MS9asZeT9rVp.jpg,1954-06-24,124,As big as the ocean!,The Caine Mutiny,7.145,306,"['Humphrey Bogart', 'Robert Francis', 'Van Joh...",['Edward Dmytryk'],"['mutiny', 'post-traumatic stress disorder (pt..."
9996,9996,False,/x4sqnmV2nAKIG6cmt9WAF9xRA2M.jpg,"['Comedy', 'Drama', 'Romance']",236,tt0110598,en,A young social outcast in Australia steals mon...,12.677,/zJyTr8Fo412a2OIfJGXTRAm4IwX.jpg,1994-09-29,106,Success is the best revenge.,Muriel's Wedding,6.868,413,"['Toni Collette', 'Bill Hunter', 'Rachel Griff...",['P.J. Hogan'],"['daughter', 'individual', 'friendship', 'drea..."
9997,9997,False,/gjysasbU9gKgbu7yW7Z1aonoiTg.jpg,"['Horror', 'Thriller']",517116,tt6535880,en,"On Halloween, a group of friends encounter an ...",12.676,/haQSrjFlq30I60UVN1VsB6uRwKM.jpg,2019-09-13,92,Some Monsters Are Real.,Haunt,6.697,1086,"['Katie Stevens', 'Will Brittain', 'Lauryn McC...",['Scott Beck'],"['mask', 'halloween', 'haunted house', 'illino..."
9998,9998,False,/j3qoYRsQueTCvFhOVakVNhqmxRW.jpg,"['Drama', 'Music']",239566,tt2473602,en,A chronicle of James Brown's rise from extreme...,12.676,/dVbGeU5agxFGcJnL5hk4uO8W7i2.jpg,2014-08-01,139,The Funk Don't Quit,Get on Up,6.763,489,"['Chadwick Boseman', 'Nelsan Ellis', 'Dan Aykr...",['Tate Taylor'],"['vietnam war', '1970s', 'biography', 'south c..."
9999,9999,False,/jRwqNjLsOiCzDJ4lT4dpFZemNx7.jpg,['Drama'],826796,tt13649036,en,Sarah seems to have found her calling working ...,12.674,/33Z4AmHQYUpMWiM0T1jYq3xPdKI.jpg,2021-09-16,98,No one's coming.,Help,7.133,98,"['Jodie Comer', 'Stephen Graham', 'Ian Hart', ...",['Marc Munden'],"['liverpool, england', 'care home', 'covid-19']"


In [12]:
#total number of rows and columns of dataset
movies.shape

(10000, 19)

In [14]:
#removing all the columns from the movie dataset except id, title, overview and genre
movies=movies[['id','title','overview','genres']]

In [15]:
#viewing top 5 rows to verify the removing the unnecessary columns
movies.head()

,id,title,overview,genres
0,572802,Aquaman and the Lost Kingdom,"Black Manta, still driven by the need to aveng...","['Action', 'Adventure', 'Fantasy']"
1,787699,Wonka,Willy Wonka – chock-full of ideas and determin...,"['Comedy', 'Family', 'Fantasy']"
2,1022796,Wish,"Asha, a sharp-witted idealist, makes a wish so...","['Animation', 'Family', 'Fantasy', 'Adventure']"
3,940551,Migration,After a migrating duck family alights on their...,"['Animation', 'Adventure', 'Comedy', 'Family',..."
4,609681,The Marvels,"Carol Danvers, aka Captain Marvel, has reclaim...","['Science Fiction', 'Adventure', 'Action']"


In [16]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        10000 non-null  int64 
 1   title     10000 non-null  object
 2   overview  9928 non-null   object
 3   genres    10000 non-null  object
dtypes: int64(1), object(3)
memory usage: 312.6+ KB


In [17]:
#checking for any duplicate rows in the dataset
movies.duplicated().sum()

np.int64(176)

In [ ]:
movies.describe()

In [19]:
#adding the genre and overview column in the tag column
movies['tag']=movies['genres']+movies['overview']

/tmp/ipykernel_4424/1268957135.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tag']=movies['genres']+movies['overview']


In [20]:
movies

,id,title,overview,genres,tag
0,572802,Aquaman and the Lost Kingdom,"Black Manta, still driven by the need to aveng...","['Action', 'Adventure', 'Fantasy']","['Action', 'Adventure', 'Fantasy']Black Manta,..."
1,787699,Wonka,Willy Wonka – chock-full of ideas and determin...,"['Comedy', 'Family', 'Fantasy']","['Comedy', 'Family', 'Fantasy']Willy Wonka – c..."
2,1022796,Wish,"Asha, a sharp-witted idealist, makes a wish so...","['Animation', 'Family', 'Fantasy', 'Adventure']","['Animation', 'Family', 'Fantasy', 'Adventure'..."
3,940551,Migration,After a migrating duck family alights on their...,"['Animation', 'Adventure', 'Comedy', 'Family',...","['Animation', 'Adventure', 'Comedy', 'Family',..."
4,609681,The Marvels,"Carol Danvers, aka Captain Marvel, has reclaim...","['Science Fiction', 'Adventure', 'Action']","['Science Fiction', 'Adventure', 'Action']Caro..."
...,...,...,...,...,...
9995,10178,The Caine Mutiny,When a US Naval captain shows signs of mental ...,"['Drama', 'War']","['Drama', 'War']When a US Naval captain shows ..."
9996,236,Muriel's Wedding,A young social outcast in Australia steals mon...,"['Comedy', 'Drama', 'Romance']","['Comedy', 'Drama', 'Romance']A young social o..."
9997,517116,Haunt,"On Halloween, a group of friends encounter an ...","['Horror', 'Thriller']","['Horror', 'Thriller']On Halloween, a group of..."
9998,239566,Get on Up,A chronicle of James Brown's rise from extreme...,"['Drama', 'Music']","['Drama', 'Music']A chronicle of James Brown's..."


In [22]:
#Now dropping the overview and genre column
new_movies=movies.drop(columns=['overview','genres'])

In [ ]:
new_movies.head()

In [23]:

# Download the stopwords data
nltk.download('stopwords')

# Get a set of common English stopwords
english_stopwords = set(stopwords.words('english'))

# Function to preprocess and remove common English stopwords from a string
def preprocess_string(input_string):
    # Tokenize the input string
    tokens = input_string.lower().split()

    # Remove common English stopwords
    tokens = [word for word in tokens if word not in english_stopwords]

    # Join the tokens back into a string
    return " ".join(tokens)

# Define a function called 'recommend' that takes a 'movies' parameter.
def recommend(movies):
    # Preprocess the input movie name
    movies = preprocess_string(movies)

    # Initialize a list to store movie indices and their similarity scores
    matching_movies = []

    # Iterate through the movie titles in the dataset and calculate a similarity score
    for index, title in enumerate(new_movies['title']):
         # Calculate a partial string matching similarity score
        similarity_score = fuzz.partial_ratio(movies, preprocess_string(title.lower()))
        matching_movies.append((index, similarity_score))

    # Filter movies with a similarity score greater than a threshold (e.g., 80)
    threshold = 80
    matching_movies = [(index, score) for index, score in matching_movies if score >= threshold]

    if not matching_movies:
        print("No matching movies found in the dataset.")
        return

    # Sort matching movies by similarity score in descending order
    matching_movies.sort(key=lambda x: x[1], reverse=True)

    # Extract the top 10 most similar movies and print their titles
    for index, _ in matching_movies[:10]:
        print(new_movies.iloc[index].title)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [24]:
# It recommends movies based on the corresponding movie genres or content based
recommend('captan ')

The Tank
Io Capitano
Captain America: Civil War
Captain Marvel
Captain America: The First Avenger
Captain America: The Winter Soldier
Catman
Captain Fantastic
Captain Phillips
Captain Underpants: The First Epic Movie


In [25]:
recommend(' Nun')

The Nun II
The Nun
Nun in Rope Hell
The Nun
The Nun
The Menu
Jamaica Inn
Dobermann


In [26]:
recommend('Inception')

Inception
Transformers: Age of Extinction
Extinction
Extinction
Perfect Addiction
The One
Creation
Stranger Than Fiction
The One


In [27]:
recommend('titani')

Titanic
Titanic II
Titanic 666
Titanic
Attack on Titan
Attack on Titan
Attack on Titan
Teen Titans: Trouble in Tokyo
Titane
Attack on Titan II: End of the World


In [28]:
recommend('')

Me Before You
Up
It
Down
Her
He's Just Not That Into You
She's All That
IF
He's All That
If Only
